# Areal maps — wells, USP, doublet layout, property contours


Three areal visualizations:
1. Well locations + USP + USP-01 proposed
2. Slochteren porosity / permeability contour maps
3. Doublet layout zoom (BLT-01 producer + USP-01 injector + USP)

## Outputs
- `05_visualizations/maps/well_locations_v01.png`
- `05_visualizations/maps/slochteren_property_contours_v01.png`
- `05_visualizations/maps/doublet_layout_v01.png`


In [ ]:
import sys
sys.path.insert(0, '../../../03_analysis/scripts')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from plotting_utils import apply_style, COLORS
apply_style()

constants = pd.read_csv('../../../02_data_processed/reservoir_summary/project_constants_v01.csv')
usp_x = constants[constants.parameter=='usp_x_rd_new'].value.values[0]
usp_y = constants[constants.parameter=='usp_y_rd_new'].value.values[0]
usp01_x = constants[constants.parameter=='usp01_x_rd_new'].value.values[0]
usp01_y = constants[constants.parameter=='usp01_y_rd_new'].value.values[0]

# Surface coordinates (RD New) and LAS-derived effective porosity (%) per well.
wells = {'BLT-01': (141577.55, 456881.76), 'EVD-01': (136997.00, 441189.00),
         'JUT-01': (134098.00, 451726.00), 'PKP-01': (118503.09, 453402.51)}
summary = pd.read_csv('../../../02_data_processed/petrophysics/slochteren_petrophysics_summary_v01.csv')
phie_pct = {r.well_id: round(r.phi_eff_frac*100, 1) for r in summary.itertuples()}

print(f"USP: ({usp_x:.0f}, {usp_y:.0f}) ; USP-01 proposed: ({usp01_x:.0f}, {usp01_y:.0f})")
for w, (x, y) in wells.items():
    print(f"  {w}: ({x:.0f}, {y:.0f})  phie={phie_pct[w]}%")

## Interpolation method for contours

`scipy.interpolate.griddata` with linear interpolation, anchored on the 4 well measurements plus the predicted USP-01 values. This is interpolation only (no extrapolation outside the well grid), so values far from wells should be treated as uncertain.

The porosity map shows a clear high near BLT-01 (11.4%) extending toward JUT-01 (10.9%) and the proposed USP-01 location (predicted 11.4%). EVD-01 and PKP-01 show much lower values — the proposed development cleanly avoids the low-porosity area.


In [ ]:
# --- 1. Well locations + USP + proposed USP-01 ---
fig, ax = plt.subplots(figsize=(9, 8))
for w, (x, y) in wells.items():
    ax.scatter(x, y, s=90, color=COLORS['shale'], zorder=3)
    ax.annotate(w, (x, y), textcoords='offset points', xytext=(8, 6), fontsize=10, fontweight='bold')
ax.scatter(usp_x, usp_y, s=170, marker='*', color=COLORS['usp'], zorder=4, label='USP (demand centre)')
ax.annotate('USP', (usp_x, usp_y), textcoords='offset points', xytext=(8, -12), fontsize=10, color=COLORS['usp'])
ax.scatter(usp01_x, usp01_y, s=120, marker='^', color=COLORS['injector'], zorder=4, label='USP-01 (proposed)')
ax.annotate('USP-01', (usp01_x, usp01_y), textcoords='offset points', xytext=(8, 6), fontsize=10, color=COLORS['injector'])
ax.set_xlabel('Easting (RD New, m)'); ax.set_ylabel('Northing (RD New, m)')
ax.set_title('Well locations and the USP', loc='left', fontweight='bold')
ax.set_aspect('equal', adjustable='datalim'); ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../../../05_visualizations/maps/well_locations_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/maps/well_locations_v01.svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- 2. Slochteren effective-porosity contour map (interpolation only, no extrapolation) ---
pts = np.array([wells[w] for w in wells] + [(usp01_x, usp01_y)])
vals = np.array([phie_pct[w] for w in wells] + [11.4])   # USP-01 predicted (IDW)
gx = np.linspace(pts[:, 0].min() - 2000, pts[:, 0].max() + 2000, 220)
gy = np.linspace(pts[:, 1].min() - 2000, pts[:, 1].max() + 2000, 220)
GX, GY = np.meshgrid(gx, gy)
GZ = griddata(pts, vals, (GX, GY), method='linear')
fig, ax = plt.subplots(figsize=(9, 8))
cf = ax.contourf(GX, GY, GZ, levels=12, cmap='YlOrBr')
cb = fig.colorbar(cf, ax=ax); cb.set_label('Effective porosity (%)')
for w, (x, y) in wells.items():
    ax.scatter(x, y, s=70, color='black', zorder=3)
    ax.annotate(f'{w}\n{phie_pct[w]}%', (x, y), textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.scatter(usp01_x, usp01_y, marker='^', s=110, color=COLORS['injector'], zorder=4)
ax.annotate('USP-01\n11.4%*', (usp01_x, usp01_y), textcoords='offset points', xytext=(6, 4),
            fontsize=8, color=COLORS['injector'])
ax.set_xlabel('Easting (RD New, m)'); ax.set_ylabel('Northing (RD New, m)')
ax.set_title('Slochteren effective porosity — interpolated from wells (* USP-01 predicted)',
             loc='left', fontweight='bold')
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.savefig('../../../05_visualizations/maps/slochteren_property_contours_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/maps/slochteren_property_contours_v01.svg', bbox_inches='tight')
plt.show()

In [ ]:
# --- 3. Doublet layout zoom (BLT-01 producer + USP-01 injector + USP plant) ---
blt = wells['BLT-01']
def _km(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2) ** 0.5 / 1000
fig, ax = plt.subplots(figsize=(9, 8))
ax.plot([blt[0], usp01_x], [blt[1], usp01_y], color=COLORS['producer'], lw=2, zorder=2)
ax.plot([usp01_x, usp_x], [usp01_y, usp_y], color=COLORS['injector'], lw=2, ls='--', zorder=2)
ax.scatter(*blt, s=170, color=COLORS['producer'], zorder=3, label='BLT-01 producer')
ax.scatter(usp01_x, usp01_y, s=160, marker='^', color=COLORS['injector'], zorder=3, label='USP-01 injector (new)')
ax.scatter(usp_x, usp_y, s=200, marker='*', color=COLORS['usp'], zorder=3, label='USP plant')
ax.annotate(f'{_km(blt, (usp01_x, usp01_y)):.1f} km', ((blt[0]+usp01_x)/2, (blt[1]+usp01_y)/2),
            fontsize=9, color=COLORS['producer'])
ax.annotate(f'{_km((usp01_x, usp01_y), (usp_x, usp_y)):.1f} km', ((usp01_x+usp_x)/2, (usp01_y+usp_y)/2),
            fontsize=9, color=COLORS['injector'])
for name, (x, y) in [('BLT-01', blt), ('USP-01', (usp01_x, usp01_y)), ('USP', (usp_x, usp_y))]:
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(8, 8), fontsize=10, fontweight='bold')
ax.set_xlabel('Easting (RD New, m)'); ax.set_ylabel('Northing (RD New, m)')
ax.set_title('Recommended doublet layout (1.5 km spacing)', loc='left', fontweight='bold')
ax.set_aspect('equal', adjustable='datalim'); ax.legend(loc='best')
plt.tight_layout()
plt.savefig('../../../05_visualizations/maps/doublet_layout_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/maps/doublet_layout_v01.svg', bbox_inches='tight')
plt.show()